# 01. 인스턴스 생성과 검증

이 notebook은 알고리즘을 구현하지 않는다. `src/cflp_bd_qa_ex2` 의 함수를 호출하고
결과를 해석한다.

확인 항목:
1. 모든 파일럿 인스턴스 생성
2. seed 와 schema 검증
3. demand / capacity / cost 분포 요약
4. 총 capacity equality assert
5. 재실행 결정성 검증

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from cflp_bd_qa_ex2.config.loader import load_config

cfg = load_config(ROOT / "config" / "experiments" / "pilot.yaml")
print("configuration_hash:", cfg["configuration_hash"][:16])


configuration_hash: 0dc9f18b09b09302


## 1. 인스턴스 생성

`facilities x customers` 순서로 표기한다. 4개 크기 x 5개 seed = 20개.

In [2]:
from cflp_bd_qa_ex2.data.generator import DRAW_ORDER, generate_all
from cflp_bd_qa_ex2.data.io import save_instance

instances = generate_all(cfg)
for inst in instances:
    save_instance(inst, ROOT / "data" / "raw")

print(f"{len(instances)}개 인스턴스 생성")
print("draw 순서 (고정):")
for i, name in enumerate(DRAW_ORDER, start=1):
    print(f"  {i}. {name}")


20개 인스턴스 생성
draw 순서 (고정):
  1. facility_locations
  2. customer_locations
  3. demand_raw
  4. capacity_raw
  5. fixed_cost_base
  6. fixed_cost_slope


## 2. 총 capacity equality assert

명세상 `sum_j s_j == ceil(1.5 * D)` 가 **정확히** 성립해야 한다.
rescale 후 floor 하고 fractional remainder 순으로 배분하므로 오차가 남으면 안 된다.

In [3]:
import math

rows = []
for inst in instances:
    target = math.ceil(1.5 * inst.total_demand)
    assert inst.total_capacity == target, f"{inst.instance_id}: 총용량 불일치"
    rows.append({
        "instance_id": inst.instance_id,
        "n_facilities": inst.n_facilities,
        "n_customers": inst.n_customers,
        "total_demand_D": inst.total_demand,
        "total_capacity": inst.total_capacity,
        "S_target": target,
        "capacity_slack_max": inst.capacity_slack_max,
        "U_obj": inst.objective_upper_bound,
    })

summary = pd.DataFrame(rows)
print("총용량 equality: 20/20 통과")
summary


총용량 equality: 20/20 통과


,instance_id,n_facilities,n_customers,total_demand_D,total_capacity,S_target,capacity_slack_max,U_obj
0,4x12_s100,4,12,421,632,632,211,13137.543834
1,4x12_s101,4,12,448,672,672,224,13713.198843
2,4x12_s102,4,12,423,635,635,212,13401.610481
3,4x12_s103,4,12,422,633,633,211,14436.545676
4,4x12_s104,4,12,421,632,632,211,13386.813191
5,8x25_s200,8,25,864,1296,1296,432,47642.654414
6,8x25_s201,8,25,906,1359,1359,453,48266.369953
7,8x25_s202,8,25,884,1326,1326,442,43242.188794
8,8x25_s203,8,25,845,1268,1268,423,48026.732504
9,8x25_s204,8,25,906,1359,1359,453,50763.441246


## 3. 분포 요약

`demand ~ N(mean=35, standard_deviation=5)` 이다. **5는 표준편차이며 분산이 아니다.**

이 생성법은 Cornuejols 계열의 위치/비용/capacity 구조를 따르되 demand 분포를 변경한
변형이다. 원래 생성법의 demand 는 후속 문헌에서 주로 `U[5,35]` 로 설명된다
(평균 20, 변동계수 약 0.43). 본 명세는 평균 35, 변동계수 약 0.14 로 **고객 간
이질성이 더 작다**. 이는 SP 의 flow 배분과 dual 구조에 영향을 줄 수 있으나,
영향의 방향과 크기는 실험적으로 평가한다.

In [4]:
dist_rows = []
for inst in instances:
    dist_rows.append({
        "instance_id": inst.instance_id,
        "demand_mean": float(inst.demand.mean()),
        "demand_std": float(inst.demand.std(ddof=1)),
        "demand_cv": float(inst.demand.std(ddof=1) / inst.demand.mean()),
        "demand_min": int(inst.demand.min()),
        "demand_max": int(inst.demand.max()),
        "capacity_mean": float(inst.capacity.mean()),
        "fixed_cost_mean": float(inst.fixed_cost.mean()),
        "transport_cost_mean": float(inst.transport_cost.mean()),
    })

dist = pd.DataFrame(dist_rows)
print("전체 demand 변동계수 평균: %.4f (Cornuejols U[5,35] 기준값 약 0.43)"
      % dist["demand_cv"].mean())
dist


전체 demand 변동계수 평균: 0.1348 (Cornuejols U[5,35] 기준값 약 0.43)


,instance_id,demand_mean,demand_std,demand_cv,demand_min,demand_max,capacity_mean,fixed_cost_mean,transport_cost_mean
0,4x12_s100,35.083333,4.851585,0.138287,26,46,158.0000,1296.285318,4.704290
1,4x12_s101,37.333333,5.694229,0.152524,28,47,168.0000,1333.402492,4.721758
2,4x12_s102,35.250000,5.154433,0.146225,22,43,158.7500,1362.608545,4.715164
3,4x12_s103,35.166667,3.242707,0.092210,29,41,158.2500,1324.079774,5.433841
4,4x12_s104,35.083333,4.055486,0.115596,26,42,158.0000,1386.497449,4.663589
5,8x25_s200,34.560000,5.331354,0.154264,27,46,162.0000,1361.736395,5.293154
6,8x25_s201,36.240000,4.780865,0.131922,23,46,169.8750,1348.932925,5.132123
7,8x25_s202,35.360000,4.707087,0.133119,24,43,165.7500,1334.167489,4.618001
8,8x25_s203,33.800000,6.157651,0.182179,23,47,158.5000,1307.193105,5.501483
9,8x25_s204,36.240000,4.867237,0.134306,27,45,169.8750,1380.700305,5.508223


## 4. Schema 검증

저장된 JSON 이 schema 를 만족하는지 확인한다.
demand 양수, 배열 shape, 총용량 일치를 모두 본다.

In [5]:
from cflp_bd_qa_ex2.data.io import instance_to_dict, load_all_instances, validate_instance_payload

loaded = load_all_instances(ROOT / "data" / "raw")
for inst in loaded:
    validate_instance_payload(instance_to_dict(inst))

print(f"schema 검증 통과: {len(loaded)}개")


schema 검증 통과: 20개


## 5. 재실행 결정성

같은 seed 로 다시 생성했을 때 **모든 배열이 정확히 일치**해야 한다.
draw 순서가 고정되어 있으므로 성립한다.

In [6]:
from cflp_bd_qa_ex2.data.generator import generate_instance

mismatches = []
for inst in instances:
    again = generate_instance(inst.name, inst.n_facilities, inst.n_customers, inst.seed, cfg)
    same = (
        np.array_equal(inst.demand, again.demand)
        and np.array_equal(inst.capacity, again.capacity)
        and np.allclose(inst.fixed_cost, again.fixed_cost)
        and np.allclose(inst.transport_cost, again.transport_cost)
        and np.allclose(inst.facility_locations, again.facility_locations)
        and np.allclose(inst.customer_locations, again.customer_locations)
    )
    if not same:
        mismatches.append(inst.instance_id)

assert not mismatches, f"결정성 위반: {mismatches}"
print("재실행 결정성: 20/20 통과")


재실행 결정성: 20/20 통과


## 해석

- 총용량 equality 가 모든 인스턴스에서 정확히 성립한다. 따라서 capacity slack 은
  `delta = 1` 의 pure exponential 로 **근사 없이** 표현된다.
- `capacity_slack_max = 0.5 * D` 수준이므로 `b_cap` 은 인스턴스 크기에 따라
  8 ~ 11 bit 범위에서 움직인다.
- `U_obj` 는 모든 `j` 에 대해 `c_ij d_i` 를 합산하므로 실제 최적값보다 훨씬 크다.
  이 느슨함이 이후 theta / cut slack 해상도에 직접 영향을 준다.